# Chg2Cap: Colab training and evaluation

Clone a GitHub branch, tag or commit, prepare LEVIR-CC, train Chg2Cap and evaluate the test split. Code and images use `/content`; checkpoints, logs and results are saved to Google Drive.

Select a GPU runtime and run the cells in order.

## 1. Configuration

Set `REPO_URL` and `REPO_REF` to a repository revision containing the training scripts. Set `DATASET_URL` to a dataset download link, or `DATASET_SOURCE` to a ZIP or folder on Google Drive. A non-empty `DATASET_SOURCE` takes precedence over the URL.

Use a unique `RUN_NAME` for each experiment. To resume, reuse the experiment name and set `RESUME=True`. `NUM_EPOCHS` is the total target epoch count, including completed epochs.

Default training settings follow the [Chg2Cap paper](https://arxiv.org/html/2304.01091#S4.SS2.SSS2): 50 epochs, batch size 32, Adam with initial learning rate 1e-4, and validation after every epoch. The repository uses StepLR with step size 5 and gamma 0.5. ResNet-101, three attentive encoder layers and one decoder layer follow the paper's architecture configuration. Dropout 0.1 and word-count threshold 5 follow the repository defaults.

Use a new experiment name and `RESUME=False` for training from scratch. The paper uses an NVIDIA A100; the assigned GPU must have sufficient memory for batch size 32.

The notebook uses initial seed 123 and stops after 10 consecutive epochs without validation BLEU-4 improvement, with a maximum of 50 epochs. Early stopping is an additional experiment setting, not part of the paper profile. Set `EARLY_STOPPING_PATIENCE=0` to disable it. A fixed seed does not guarantee identical results across runtimes or resumed runs.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import subprocess
import shutil
import zipfile

REPO_URL = "https://github.com/yakupgulcan/Chg2Cap.git"
REPO_REF = "feature/modernize-runtime"  # Branch, tag or commit SHA
PROJECT_DIR = Path("/content/Chg2Cap")
DATASET_URL = "https://drive.google.com/file/d/1YgWES-0OOL-3KK4yIV3-y8MN-_c1ELEE/view?usp=drive_link"
DATASET_SOURCE = ""  # e.g. /content/drive/MyDrive/datasets/LEVIR_CC.zip
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/Chg2Cap_Outputs")
RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
NUM_EPOCHS = 50
SEED = 123
EARLY_STOPPING_PATIENCE = 10  # 0 disables early stopping
TRAIN_BATCHSIZE = 32
WORKERS = 2
ENCODER_LR = 1e-4
DECODER_LR = 1e-4
NETWORK = "resnet101"
ENCODER_LAYERS = 3
DECODER_LAYERS = 1
ATTENTION_HEADS = 8
HIDDEN_DIM = 512
FEATURE_DIM = 2048
DROPOUT = 0.1
RESUME = False
RUN_SMOKE_CHECK = True

if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 or newer is required by this notebook.")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME:
    raise ValueError("RUN_NAME must be one directory name.")

## 2. Mount Drive and clone the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} exists. Use a fresh runtime or choose a new project directory.")
subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=PROJECT_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
for required in ["requirements-colab.txt", "runtime.py", "drive_checkpoints.py", "scripts/prepare_meteor.py", "scripts/smoke_check.py"]:
    if not (PROJECT_DIR / required).is_file():
        raise FileNotFoundError(f"Selected Git ref does not contain {required}. Choose a repository revision containing the required scripts.")
RUN_DIR = DRIVE_OUTPUT_ROOT / RUN_NAME
if RUN_DIR.exists() and not RESUME:
    raise FileExistsError("Run already exists. Choose a new RUN_NAME or set RESUME=True.")
if RESUME and not RUN_DIR.is_dir():
    raise FileNotFoundError("Resume experiment directory is missing.")
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = PROJECT_DIR / "runs" / RUN_NAME / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = RUN_DIR / "test_results"
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3. Dependencies and environment

Keep Colab's installed PyTorch, torchvision and CUDA build. Install the additional dependencies, prepare the METEOR resource and check package imports, GPU access and Java execution.

Environment versions, installed packages and dependency checks are saved to the experiment directory. If packages have already been imported before installation, restart the runtime before proceeding.

In [ ]:
def run_checked(command, label):
    result = subprocess.run(
        command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    )
    print(f"--- {label} ---", flush=True)
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n", flush=True)
    if result.returncode:
        raise RuntimeError(f"{label} failed (exit {result.returncode}). See the command output above.")
    return result

run_checked(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt", "gdown==5.2.0", "jedi>=0.16,<0.20", "google-api-python-client>=2,<3", "google-auth>=2,<3"],
    "Install dependencies",
)
if shutil.which("java") is None:
    run_checked(["apt-get", "update", "-qq"], "Update package index")
    run_checked(["apt-get", "install", "-y", "openjdk-17-jre-headless"], "Install Java")
run_checked([sys.executable, "scripts/prepare_meteor.py"], "Prepare METEOR")

run_checked([sys.executable, "-u", "-c", """
import sys
import torch
print('Python:', sys.version, flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('CUDA build:', torch.version.cuda, flush=True)
import torchvision
print('torchvision:', torchvision.__version__, flush=True)
"""], "Package imports")

run_checked([sys.executable, "-u", "-c", """
import torch
print('CUDA available:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. Select a GPU in Runtime > Change runtime type.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
x = torch.ones(4, device='cuda')
value = (x + x).sum().item()
if value != 8:
    raise RuntimeError(f'Unexpected GPU operation result: {value}')
print('GPU operation: passed', flush=True)
"""], "GPU access")

run_checked([sys.executable, "-u", "-c", """
from eval_func.meteor.meteor import Meteor
metric = Meteor()
try:
    score, _ = metric.compute_score([['1 2 3']], [['1 2 3']])
    print('METEOR score:', score, flush=True)
finally:
    metric.close()
"""], "METEOR execution")

from runtime import environment_info, write_json
record_name = datetime.now(timezone.utc).strftime("environment_%Y%m%d_%H%M%S")
write_json(RUN_DIR / (record_name + ".json"), environment_info())
packages = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / (record_name + "_packages.txt")).write_text(packages)
package_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], cwd=PROJECT_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
(RUN_DIR / (record_name + "_pip_check.txt")).write_text(package_check.stdout)
print(package_check.stdout)
if package_check.returncode:
    raise RuntimeError("Dependency conflicts found. See the pip check output above.")
print(json.dumps(environment_info(), indent=2))


## Checkpoint storage

Authorize Drive API access to upload and replace checkpoints in the selected experiment folder. On resume, existing checkpoints are downloaded to runtime storage and verified before loading. The output root must be under `MyDrive`.


In [ ]:
from google.colab import auth
from drive_checkpoints import DriveCheckpoints, drive_service

auth.authenticate_user()
service = drive_service()

def drive_folder(parent_id, name):
    escaped = name.replace("\\", "\\\\").replace("'", "\\'")
    matches = service.files().list(
        q=f"'{parent_id}' in parents and name = '{escaped}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
        fields="files(id)", pageSize=100,
    ).execute().get("files", [])
    if len(matches) > 1:
        raise RuntimeError(f"Multiple Drive folders named {name}; select a unique experiment path.")
    if matches:
        return matches[0]["id"]
    if RESUME:
        raise FileNotFoundError(f"Resume Drive folder missing: {name}")
    return service.files().create(
        body={"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]},
        fields="id",
    ).execute()["id"]

relative = RUN_DIR.relative_to(Path("/content/drive/MyDrive"))
folder_id = "root"
for part in (*relative.parts, "checkpoints"):
    folder_id = drive_folder(folder_id, part)
DRIVE_CHECKPOINT_FOLDER_ID = folder_id
store = DriveCheckpoints(folder_id, service)
if RESUME:
    store.download("last.pth", CHECKPOINT_DIR / "last.pth")
    if store.files("best.pth"):
        store.download("best.pth", CHECKPOINT_DIR / "best.pth")
print("Drive checkpoint folder:", DRIVE_CHECKPOINT_FOLDER_ID)


## 4. Prepare LEVIR-CC

The dataset must contain `LevirCCcaptions.json` and `images/{train,val,test}/{A,B}`. ZIP archives with one enclosing folder are supported. Images are copied to runtime storage.

Preprocessing checks image paths and writes caption tokens, split lists and a vocabulary with word-count threshold 5. Repeated preprocessing rewrites split lists without duplicate rows.

In [ ]:
DATASET_ROOT = PROJECT_DIR / "datasets/LEVIR_CC"
if not DATASET_ROOT.exists():
    if DATASET_SOURCE:
        source = Path(DATASET_SOURCE)
        if not source.exists():
            raise FileNotFoundError(source)
    else:
        if not DATASET_URL:
            raise ValueError("Set DATASET_SOURCE or DATASET_URL.")
        import gdown
        source = Path("/content/LEVIR_CC_download.zip")
        if gdown.download(DATASET_URL, str(source), fuzzy=True) is None:
            raise RuntimeError("Dataset download failed.")
    if source.is_file():
        staging = Path("/content/Chg2Cap_dataset_extract")
        if staging.exists():
            raise FileExistsError("Dataset extraction staging exists; use a fresh runtime.")
        staging.mkdir()
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                target = (staging / member.filename).resolve()
                if not target.is_relative_to(staging.resolve()):
                    raise ValueError("Archive member escapes the extraction directory.")
            archive.extractall(staging)
        candidates = list(staging.rglob("LevirCCcaptions.json"))
    else:
        candidates = list(source.rglob("LevirCCcaptions.json"))
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly one captions JSON, found {len(candidates)}.")
    dataset_folder = candidates[0].parent
    if not (dataset_folder / "images").is_dir():
        raise FileNotFoundError("images directory must be next to LevirCCcaptions.json")
    DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(dataset_folder, DATASET_ROOT)

PREPARED_DIR = PROJECT_DIR / "data/LEVIR_CC"
subprocess.run([sys.executable, "preprocess_data.py", "--dataset", "LEVIR_CC", "--dataset_root", str(DATASET_ROOT), "--word_count_threshold", "5"], check=True)
manifest = json.loads((PREPARED_DIR / "manifest.json").read_text())
if manifest["max_length"] > 41:
    raise ValueError("Captions exceed the default max_length=41; configure train and test consistently.")
archive_dir = RUN_DIR / "prepared"
if RESUME:
    for name in ["train.txt", "val.txt", "test.txt", "vocab.json", "manifest.json"]:
        if (archive_dir / name).read_bytes() != (PREPARED_DIR / name).read_bytes():
            raise ValueError(f"Prepared {name} differs from the saved run.")
else:
    archive_dir.mkdir()
    for name in ["train.txt", "val.txt", "test.txt", "vocab.json", "manifest.json"]:
        shutil.copy2(PREPARED_DIR / name, archive_dir / name)
print(manifest)

## 5. Optional short run

When `RUN_SMOKE_CHECK=True`, run two training batches, four validation pairs, epoch resume and four test pairs. These outputs are stored separately from the main experiment.

In [ ]:
def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + json.dumps(command) + "\n")
        log.flush()
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            return_code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
        finally:
            process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

if RUN_SMOKE_CHECK:
    smoke_dir = PROJECT_DIR / "runs" / ("smoke_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"))
    run_logged([sys.executable, "-u", "scripts/smoke_check.py", "--dataset_root", str(DATASET_ROOT), "--output_dir", str(smoke_dir), "--resume_check"], RUN_DIR / "smoke.log")

## 6. Train

`last.pth` contains model, optimizer and scheduler states at completed training epochs. `best.pth` contains inference weights selected by validation BLEU-4.

Resume starts at the next epoch. Within-epoch progress and random generator states are not restored: batch shuffling and dropout can differ from uninterrupted training, so an identical numerical trajectory is not guaranteed. Learned weights and optimizer/scheduler states are retained. If validation is interrupted, that epoch's validation is not replayed.

Checkpoints are saved on runtime storage, then uploaded through the Drive API. After size and checksum verification, the previous checkpoint of the same name in this experiment's checkpoint folder is permanently deleted, without entering the trash. Upload or deletion errors stop training. Keep free space for both old and new copies during upload. Existing trash is not removed. Use a separate experiment directory for each training process.

Use a new experiment name for fresh training. Reduce the training batch size if CUDA runs out of memory. Smoke-check checkpoints remain on runtime storage; logs are saved to Drive.

The early-stopping counter and best epoch are saved in `last.pth`. Older checkpoints without the counter start it at zero. Each epoch records the learning rates used for training in `training_epoch_XXX.json` and the training log. `best.pth` remains the checkpoint selected for final test evaluation.


In [ ]:
model_options = [
    "--network", NETWORK,
    "--n_layers", str(ENCODER_LAYERS),
    "--decoder_n_layers", str(DECODER_LAYERS),
    "--n_heads", str(ATTENTION_HEADS),
    "--hidden_dim", str(HIDDEN_DIM),
    "--encoder_dim", str(FEATURE_DIM),
    "--attention_dim", str(FEATURE_DIM),
    "--feature_dim", str(FEATURE_DIM),
    "--dropout", str(DROPOUT),
]
command = [
    sys.executable, "-u", "train.py",
    "--data_folder", str(DATASET_ROOT / "images"),
    "--savepath", str(CHECKPOINT_DIR),
    "--drive_checkpoint_folder_id", DRIVE_CHECKPOINT_FOLDER_ID,
    "--num_epochs", str(NUM_EPOCHS),
    "--seed", str(SEED),
    "--patience", str(EARLY_STOPPING_PATIENCE),
    "--train_batchsize", str(TRAIN_BATCHSIZE),
    "--workers", str(WORKERS),
    "--encoder_lr", str(ENCODER_LR),
    "--decoder_lr", str(DECODER_LR),
    "--fine_tune_encoder", "true",
    "--print_freq", "100",
    *model_options,
]
if RESUME:
    command += ["--checkpoint", str(CHECKPOINT_DIR / "last.pth")]
write_json(RUN_DIR / "notebook_config.json", {
    "repo_url": REPO_URL, "repo_ref": REPO_REF,
    "dataset_url": DATASET_URL, "dataset_source": DATASET_SOURCE,
    "run_name": RUN_NAME, "epochs": NUM_EPOCHS,
    "seed": SEED, "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "batchsize": TRAIN_BATCHSIZE, "workers": WORKERS, "resume": RESUME,
    "encoder_lr": ENCODER_LR, "decoder_lr": DECODER_LR,
    "network": NETWORK, "encoder_layers": ENCODER_LAYERS,
    "decoder_layers": DECODER_LAYERS, "attention_heads": ATTENTION_HEADS,
    "hidden_dim": HIDDEN_DIM, "feature_dim": FEATURE_DIM, "dropout": DROPOUT,
    "lr_schedule": {"step_size": 5, "gamma": 0.5},
})
try:
    run_logged(command, RUN_DIR / "train.log")
finally:
    for record in CHECKPOINT_DIR.glob("*.json"):
        shutil.copy2(record, RUN_DIR / record.name)


## 7. Evaluate the test split

Evaluate `best.pth` on the held-out test split. The evaluator uses one image pair per batch and saves overall, change and no-change metrics, generated captions and environment information.

Re-running evaluation in the same results directory replaces its JSON files and appends to the log.

In [ ]:
EVAL_CHECKPOINT = "best.pth"
if not (CHECKPOINT_DIR / EVAL_CHECKPOINT).is_file():
    raise FileNotFoundError(CHECKPOINT_DIR / EVAL_CHECKPOINT)
run_logged([sys.executable, "-u", "test.py", "--data_folder", str(DATASET_ROOT / "images"), "--savepath", str(CHECKPOINT_DIR), "--checkpoint", EVAL_CHECKPOINT, "--workers", str(WORKERS), "--results_dir", str(RESULTS_DIR), *model_options], RUN_DIR / "test.log")
print(json.dumps(json.loads((RESULTS_DIR / "metrics.json").read_text()), indent=2))